In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
DATA_ROOT = os.environ.get("CKD_DATA_DIR", "data")
df = pd.read_csv(os.path.join(DATA_ROOT, "kidney_disease.csv"))

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
names = pd.read_csv(os.path.join(DATA_ROOT, "data_description.txt"),sep = "-")
names = names.reset_index()
names.columns = ['cols','abb_col_names']

In [ ]:
names

In [ ]:
# Renaming the columns names for analysis
df.columns = names['abb_col_names'].values

In [ ]:
df.head()

In [ ]:
# Now changing the types of the some object features to numeric types
df.dtypes

As it can be seen, red_blood_cell_count, packed_cell_volume and white_blood_cell_count are object type. We need to 
change to numerical dtype.

In [ ]:
features=['red blood cell count','white blood cell count','packed cell volume']

In [ ]:
df['red blood cell count'] = pd.to_numeric(df['red blood cell count'],errors='coerce')

In [ ]:
def convert_dtype(df,feature):
    df[feature] = pd.to_numeric(df[feature],errors = 'coerce') # coerce will fix the errors

In [ ]:
for feature in features:
    convert_dtype(df,feature)

In [ ]:
df.dtypes

In [ ]:
# dropping id column
df.drop(['id'],axis = 1, inplace=True)

# Extracting Numberical and Categorical Features

In [ ]:
def extract_cat_num(df):
    cat_col = [col for col in df.columns if df[col].dtype == 'object']
    num_col = [col for col in df.columns if df[col].dtype != 'object']
    return cat_col, num_col

In [ ]:
cat_col, num_col = extract_cat_num(df)

In [ ]:
cat_col

In [ ]:
num_col

### To Check dirtyness in the data 

In [ ]:
for col in cat_col:
    print('{} has {} values'.format(col,df[col].unique()))

## Dirtyness in data
     ckd-chronic kidney disease
     notckd- not crornic kidney disease

In [ ]:
# Replace incorrect values
df['diabetes mellitus'].replace(to_replace = {'\tno':'no', '\tyes':'yes',' yes':'yes'},inplace=True)
df['coronary artery disease'].replace(to_replace = {'\tno':'no', '\tyes':'yes'},inplace=True)
df['class'].replace(to_replace={'ckd\t':'ckd'},inplace=True)

In [ ]:
for col in cat_col:
    print('{} has {} values'.format(col,df[col].unique()))

## Checking features distribution

In [ ]:
plt.figure(figsize=(30,30))
for i , feature in enumerate(num_col):
    plt.subplot(5,3,i+1)
    df[feature].hist()
    plt.title(feature)

    Observations:
        1.age looks a bit left skewed
        2.Blood gluscose random is right skewed
        3.Blood Urea is also a bit right skewed
        4.Rest of the features are lightly skewed

### Now, let's check the label distribution for categorical data

In [ ]:
len(cat_col)

In [ ]:
plt.figure(figsize=(20,20))
for i,data in enumerate(cat_col):
    plt.subplot(4,3,i+1)
    sns.countplot(df[data])

    fig = plt.figure(figsize=(8,8))   #  figure size (optional)

    plt.subplot(a,b,c)                #  subplot 1
    plt.plot(data_1)                  #  data 1

    plt.subplot(a,b,c)                #  subplot 2
    plt.plot(data_2)                  #  data 2

    plt.show()                        #   draw subplot group

    where:
    a  - number of rows
    b  - number of columns
    c  - plot counter

    Three Rules for Subplots:
    1.row (a) and column (b) numbers are constant per subplot group
    2.plot counter (c) is initialized and then incremented for each subplot
    3.subplot(abc) and subplot(a,b,c) are the same commands

A few features have imbalanced categories. Stratified folds will be necessary while cross validation.

In [ ]:
sns.countplot(x='class',data=df)
plt.xlabel("classification")
plt.ylabel("Count")
plt.title("target Class")

# Correlations

In [ ]:
plt.figure(figsize = (20,20))
corr_df = df.corr()
sns.heatmap(corr_df,annot=True,cmap='RdYlGn')

In [ ]:
plt.figure(figsize = (20,20))
sns.heatmap(df.corr(),annot=True)

In [ ]:
df.corr()

In [ ]:
df.groupby(['red blood cells','class'])['red blood cell count'].agg(['count','mean','median','min','max'])

## let's check for Positive correlation and its impact on classes

In [ ]:
import plotly.express as px

In [ ]:
px.violin(df, y='red blood cell count',x="class",color="class")

In [ ]:
px.scatter(df,'blood glucose random','sugar')

In [ ]:
px.scatter(df,'haemoglobin','packed cell volume')

In [ ]:
px.scatter(df,'haemoglobin','red blood cell count')

In [ ]:
### analysing distribution of 'red_blood_cell_count' in both Labels 

grid=sns.FacetGrid(df, hue="class",aspect=2)
grid.map(sns.kdeplot, 'red blood cell count')
grid.add_legend()

    Both distributions are quite different, distribution CKD is quite normal and evenly distributed but not CKD distribution is a little bit left-skewed but quite close to a normal distribution

In [ ]:
grid = sns.FacetGrid(df, hue = 'class',aspect=2)
grid.map(sns.kdeplot, 'haemoglobin')
grid.add_legend()

Haemoglobing Count is bit less in ckd compare to not ckd

In [ ]:
# Defining violin and scatter plot & kde_plot functions
def violin(col):
    fig = px.violin(df, y=col, x='class',color='class',box=True)
    return fig.show()

In [ ]:
def scatters(col1, col2):
    fig = px.scatter(df, x=col1, y=col2, color='class')
    return fig.show()

In [ ]:
def kde_plot(feature):
    grid = sns.FacetGrid(df, hue="class",aspect=2)
    grid.map(sns.kdeplot,feature)
    grid.add_legend()

In [ ]:
scatters('haemoglobin','red blood cell count')

In [ ]:
for feature in num_col:
    kde_plot(feature)

In [ ]:
for feature in num_col:
    violin(feature)

In [ ]:
def scatters(col1, col2):
    fig = px.scatter(df, x=col1, y=col2, color='class')
    return fig.show()

In [ ]:
len(num_col)

In [ ]:
scatters('white blood cell count','red blood cell count')

In [ ]:
scatters('haemoglobin','packed cell volume')

In [ ]:
cat_col

In [ ]:
num_col

     1.RBC count range ~2 to <4.5 and Hemoglobin between 3 to <13 are mostly classified as positive for chronic kidney  
        disease(i.e ckd).
     2.RBC count range >4.5 to ~6.1 and Hemoglobin between >13 to 17.8 are classified as negative for chronic kidney 
        disease(i.e nockd).

###  Now let's check for negative correlation and its impact on classes

In [ ]:
scatters('red blood cell count','albumin')

### Clearly, albumin levels of above 0 affect ckd largely

In [ ]:
scatters('packed cell volume','blood urea')

### Packed cell volume >= 40 largely affects to be non ckd

In [ ]:
fig = px.bar(df, x="specific gravity", y="packed cell volume",
             color='class', barmode='group',height=400)
fig.show()

### Clearly, specific gravity >=1.02 affects non ckd

In [ ]:
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
### filling missing with Random value

In [ ]:
cat_col

In [ ]:
num_col

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# for col in cat_col:
#     data[col]=le.fit_transform(data[col])
# for col in cat_col:
#     print('{} has {} categories'.format(col, data[col].nunique()))

# Random Value Implutation

In [ ]:
df['blood pressure'].isnull().sum()

In [ ]:
sns.countplot(df['red blood cells'])

In [ ]:
data=df.copy()

### filling missing with Random value

In [ ]:
data['red blood cells'].isnull().sum()

In [ ]:
data['red blood cells'].dropna().sample()

In [ ]:
data['red blood cells'].unique()

In [ ]:
random_sample=data['red blood cells'].dropna().sample(data['red blood cells'].isnull().sum())
random_sample

In [ ]:
random_sample.index

In [ ]:
data[data['red blood cells'].isnull()].index

In [ ]:
random_sample.index=data[data['red blood cells'].isnull()].index

In [ ]:
random_sample.isnull().sum()

In [ ]:
data.loc[data['red blood cells'].isnull(),'red blood cells']=random_sample

In [ ]:
data.head()

In [ ]:
sns.countplot(data['red blood cells'])

### lets create a function so that I can easily do it for all features

In [ ]:
def Random_value_imputation(feature):
    random_sample = data[feature].dropna().sample(data[feature].isnull().sum())
    random_sample.index = data[data[feature].isnull()].index
    data.loc[data[feature].isnull(),feature] = random_sample 

In [ ]:
Random_value_imputation('pus cell')

In [ ]:
data['pus cell'].isnull().sum()

In [ ]:
data[cat_col].isnull().sum()

In [ ]:
df[num_col].isnull().sum()

# How will we handle above data missing? 
### Numerical values using Random Value Imputation and Categorial some using Random Value imputation and rest categorial values to using mode

### as rest of the features has less missing values,so I can fill it using mode concept

In [ ]:
mode = data['pus cell clumps'].mode()[0]
mode

In [ ]:
def impute_mode(feature):
    mode = data[feature].mode()[0]
    data[feature] = data[feature].fillna(mode)

In [ ]:
for col in cat_col:
    impute_mode(col)

In [ ]:
data[cat_col].isnull().sum()

In [ ]:
data[num_col].isnull().sum()

### lets fill missing values in Numerical features using Random value Imputation

In [ ]:
for col in num_col:
    Random_value_imputation(col)

In [ ]:
data[num_col].isnull().sum()

# Feature Encoding

In [ ]:
for col in cat_col:
    print('{} has {} categories'.format(col, data[col].nunique()))

#### as we have just 2 categories in each feature then we can consider Label Encoder as it will not cause Curse of Dimensionality . It is use to convert categorial values to numerical values i.e. inform of 1's and 0's

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in cat_col:
    data[col]=le.fit_transform(data[col])

In [ ]:
data.head()

# Feature Importance

# SelectKBest-to select k best features

chi2-Internally this class is going to check that whether p-value is less than 0.05 or notbased on that,it will actually order all the features

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

In [ ]:
ind_col = [col for col in data.columns if col!='class'] # simply removing the last col from the features for training set
dep_col = 'class' # only target Column

In [ ]:
X = data[ind_col]
y = data[dep_col]

In [ ]:
order_rank_features = SelectKBest(score_func=chi2, k=20)
ordered_feature = order_rank_features.fit(X,y)

In [ ]:
ordered_feature

In [ ]:
# To get scores(rank) of feature,what we can do we can use scores function
ordered_feature.scores_

In [ ]:
datascores = pd.DataFrame(ordered_feature.scores_, columns = ['Scores'])
datascores

In [ ]:
dfcolumns=pd.DataFrame(X.columns)
dfcolumns

In [ ]:
features_rank = pd.concat([dfcolumns, datascores],axis = 1)

In [ ]:
features_rank

In [ ]:
features_rank.columns=['Features','Score']
features_rank

In [ ]:
# fetch largest 10 values of Score column
features_rank.nlargest(10,'Score')

In [ ]:
selected_columns=features_rank.nlargest(10,'Score')['Features'].values

In [ ]:
X_new = data[selected_columns]

In [ ]:
X_new

# Model building

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_new,y,train_size=0.75, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
## check whether dataset is imbalance or not
y_train.value_counts()

### Lets find best model using Hyperparameter optimization

In [ ]:
from xgboost import XGBClassifier
XGBClassifier()

In [ ]:
params={
 "learning_rate"    : [0.05, 0.20, 0.25 ] ,
 "max_depth"        : [ 5, 8, 10, 12],
 "min_child_weight" : [ 1, 3, 5, 7 ],
 "gamma"            : [ 0.0, 0.1, 0.2 , 0.4 ],
 "colsample_bytree" : [ 0.3, 0.4, 0.7 ]
    
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from xgboost import XGBClassifier
classifier=XGBClassifier()

In [ ]:
import warnings
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
random_search=RandomizedSearchCV(classifier,param_distributions=params,n_iter=5,scoring='roc_auc',n_jobs=-1,cv=5,verbose=3)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
random_search.best_estimator_

In [ ]:
random_search.best_params_

In [ ]:
classifier = XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=0.3,
              enable_categorical=False, gamma=0.1, gpu_id=-1,
              importance_type=None, interaction_constraints='',
              learning_rate=0.2, max_delta_step=0, max_depth=5,
              min_child_weight=1, monotone_constraints='()',
              n_estimators=100, n_jobs=8, num_parallel_tree=1, predictor='auto',
              random_state=0, reg_alpha=0, reg_lambda=1, scale_pos_weight=1,
              subsample=1, tree_method='exact', validate_parameters=1,
              verbosity=None)

we have got this model on the basis of cross valudation & hyper-parameter optimization

In [ ]:
classifier.fit(X_train, y_train)

In [ ]:
y_pred = classifier.predict(X_test)

In [ ]:
y_pred # this is perdicted values

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
confusion = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(confusion)

In [ ]:
### to make confusion matrix user-friendly
plt.imshow(confusion)

In [ ]:
accuracy_score(y_test, y_pred)

# Accuracy : XGBoost 98% 

# For Deploying the model

In [ ]:
# Install joblib from requirements.txt before running; notebook execution should not install packages.


In [ ]:
import joblib

In [ ]:
from joblib import dump, load
dump(classifier, 'kidney_Model.joblib')

# Testing the model on test data

In [ ]:
from joblib import dump, load
model = load('kidney_Model.joblib')
# features = np.array([[5800.,56.0,234.0,28.0,1.9,1.0,10.,73.,4.0,0]])
# features = np.array([[7700.0,26.0,116.0,45.0,1.0,0.0,15.8,53.0,0.0,0]])
features = np.array([[8100.0,106.0,27.0,0.7,42.0,0.0,14.4,64.0,0.0,0]])

results_pred = int(model.predict(features))
print(results_pred)
# 1 = ture Person having Chronical Kidney Diesease | 0 = false Not Chronical Kidney Disease
if(results_pred):
    print("There are chances of Chronic Kidney Disease ! Consult your doctor Soon.")
else:
    print("No Kidney Disease Chances")

In [ ]:
X_test

In [ ]:
y_test